The interview answer

If someone asks:

Is a Delta table a Parquet file?

A good answer is:

A Delta table stores its data as Parquet files, but it also maintains a _delta_log directory that tracks transactions and metadata. This additional transaction log enables features such as ACID transactions, Time Travel, Schema Enforcement, and Schema Evolution, which are not available with plain Parquet files.

In [0]:
data = [
    (1, "Alice", 70000),
    (2, "Bob", 50000),
    (3, "Charlie", 80000)
]

df = spark.createDataFrame(data, ["id", "name", "salary"])

display(df)

In [0]:
# df.write.format("delta").mode("overwrite").save("/tmp/day3_delta")
df.write.mode("overwrite").saveAsTable("day3_employee")

In [0]:
display(spark.table("day3_employee"))

In [0]:
%sql
DESCRIBE DETAIL day3_employee;

In [0]:
%sql
DESCRIBE HISTORY day3_employee;

In [0]:
new_data = [
    (1, "Alice", 90000),
    (2, "Bob", 55000),
    (3, "Charlie", 85000)
]

spark.createDataFrame(new_data, ["id", "name", "salary"]) \
     .write.mode("overwrite") \
     .saveAsTable("day3_employee")

In [0]:
%sql
DESCRIBE HISTORY day3_employee;

In [0]:
%sql
SELECT * FROM day3_employee VERSION AS OF 0;

In [0]:
wrong_data = [
    (4, "David", "Ninety Thousand")
]

wrong_df = spark.createDataFrame(
    wrong_data,
    ["id", "name", "salary"]
)

In [0]:
wrong_df.write.mode("append").saveAsTable("day3_employee")

In [0]:
new_data = [
    (4, "David", 60000, "IT"),
    (5, "John", 75000, "HR")
]

new_df = spark.createDataFrame(
    new_data,
    ["id", "name", "salary", "department"]
)

In [0]:
new_df.write.mode("append").saveAsTable("day3_employee")

In [0]:
new_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("day3_employee")

In [0]:
%sql
SELECT * FROM day3_employee;

First of all, well done. 👏 I can clearly see you've understood the big picture of Delta Lake. A few answers need refinement, but you're on the right track. If this were a beginner Databricks interview, I'd say you're around 8/13. After a few corrections, you'll easily reach 12/13.

Let's review each answer.

Q1. What is Delta Lake?
Your Answer

This is a part of a lakehouse, it is located on top of data lake...

Score: ⭐⭐⭐⭐☆ (4/5)

You're thinking in the right direction. The key improvement is this:

Delta Lake is an open-source storage layer built on top of Parquet files that adds ACID transactions, transaction logs, schema enforcement, schema evolution, and time travel.

A Lakehouse is an architecture. Delta Lake is one of the technologies that makes the Lakehouse possible.

Q2. Why was Delta Lake created?
Your Answer

parquet is stored and much faster, databricks makes them smarter...

Score: ⭐⭐⭐☆☆ (3/5)

You got the idea but missed the actual problems.

A better interview answer:

Parquet stores data efficiently, but it lacks:

ACID transactions
Time Travel
Schema Enforcement
Schema Evolution
Transaction History
Reliable concurrent updates

Delta Lake adds these capabilities while still storing the data as Parquet files.

Q3. Is Delta Lake a file format?
Your Answer

No, it is a technology...

Score: ⭐⭐⭐⭐⭐ (5/5)

Perfect.

I would only say:

Delta Lake is a storage layer built on top of Parquet.

Q4. Where is the data stored?
Your Answer

Data lake

Score: ⭐⭐⭐⭐☆ (4/5)

Almost.

A more complete answer:

Actual data → Parquet files
Transaction history → _delta_log
Both are stored in cloud storage (ADLS/S3/GCS) or managed storage.
Q5. ACID
Your Answer

You expanded it correctly.

Score: ⭐⭐⭐☆☆ (3/5)

Interviewers usually want explanations too.

Example:

Atomicity

Either everything succeeds or nothing does.

Transfer ₹100:

A -100
B +100

If the second step fails,

Delta rolls back the first one.

Consistency

The database always remains valid.

No corrupted or half-written data.

Isolation

Two users writing simultaneously don't interfere with each other.

Durability

Once committed,

even after a crash,

the data remains.

Q6. Two users update salary simultaneously
Your Answer

no idea

Score: ⭐☆☆☆☆

Answer:

Delta uses optimistic concurrency control.

Example:

Employee salary = 50,000

User A changes to 55,000.

User B changes to 60,000.

Delta checks whether the table changed before committing.

If another transaction already modified the same data, one commit succeeds and the other receives a conflict instead of silently overwriting data.

This protects data consistency.

Q7. Time Travel
Your Answer

get old data

Score: ⭐⭐⭐⭐☆

Correct.

A better interview answer:

Time Travel lets us query previous versions of a Delta table.

Use cases:

Recover deleted data
Audit history
Compare versions
Debug pipelines
Q8. Recover after DELETE
Your Answer

we can get it in old version

Score: ⭐⭐⭐⭐☆

Correct.

You could also mention:

SELECT * FROM employee VERSION AS OF 2;
Q9. Schema Enforcement
Your Answer

if we insert id string instead of int

Score: ⭐⭐⭐⭐⭐

Excellent.

That's exactly the idea.

Q10. Schema Evolution
Your Answer

Add the column with mergeSchema

Score: ⭐⭐⭐⭐⭐

Perfect.

Q11. What happens after every write?
Your Answer

no idea

Score: ⭐☆☆☆☆

This is important.

Every write creates a new transaction.

Delta updates _delta_log.

For example:

Write #1
000000.json

Write again

000001.json

Write again

000002.json

Each new version records:

what files were added,
what files were removed,
metadata,
the timestamp,
the operation (WRITE, UPDATE, DELETE, MERGE, etc.).

That's how Time Travel and transaction history work.

Q12. Can you see _delta_log in Free Edition?
Your Answer

Correct.

Score: ⭐⭐⭐⭐⭐

Excellent.

Q13. Fill in the blanks

You wrote:

Delta Lake is built on top of Data lake files...

Almost.

Correct answer:

Delta Lake is built on top of Parquet files and adds a transaction log (_delta_log) to provide ACID transactions, Time Travel, and Schema Evolution.

Score: ⭐⭐⭐⭐☆

Overall score
Question	Score
Q1	4/5
Q2	3/5
Q3	5/5
Q4	4/5
Q5	3/5
Q6	1/5
Q7	4/5
Q8	4/5
Q9	5/5
Q10	5/5
Q11	1/5
Q12	5/5
Q13	4/5

Approximate understanding: 82–85% ✅

That's a solid foundation for Day 3.

A few interview tips

There are three sentences I'd like you to remember because they come up often in interviews:

Delta Lake stores data in Parquet files and stores transaction history in the _delta_log directory.
Delta Lake is not a file format; it is a storage layer built on top of Parquet.
Every write operation creates a new version in the transaction log, enabling ACID transactions and Time Travel.
✅ You're ready for Day 4

Since you've grasped the fundamentals of Delta Lake, the next logical topic is Delta Lake DML Operations:

UPDATE
DELETE
MERGE (Upsert) — one of the most frequently asked Databricks interview topics
How Delta handles these operations internally

MERGE is especially important because it's used extensively in real ETL and data engineering pipelines.

Delta Lake is built on top of Parquet files and adds a transaction log (_delta_log) to provide ACID transactions, Time Travel, and Schema Evolution.
so, the parquet will be in data lake? am i right?

Yes, you're almost exactly right. 👍

The only thing I'd change is the wording.

Instead of saying:

Parquet will be in the Data Lake

Say:

Parquet files are stored in the Data Lake. Delta Lake is built on top of those Parquet files.

Think of it like this:

                Lakehouse
                    │
                    ▼
            Delta Lake (Storage Layer)
                    │
      ┌─────────────┴─────────────┐
      ▼                           ▼
  Parquet Files              _delta_log
 (Actual Data)          (Transaction Log)
      │
      └──────────────┐
                     ▼
          Data Lake Storage
      (ADLS / S3 / GCS / Managed Storage)
In Azure Databricks

Suppose your storage account is:

abfss://datalake@storageaccount.dfs.core.windows.net/

Inside it, your table might look like:

abfss://datalake@storageaccount.dfs.core.windows.net/

employee/
│
├── part-00000.snappy.parquet
├── part-00001.snappy.parquet
├── part-00002.snappy.parquet
│
└── _delta_log/
      ├── 00000000000000000000.json
      ├── 00000000000000000001.json
      └── ...

Both the Parquet files and the _delta_log are stored in the Data Lake.

One small correction

You said earlier:

"If we query, the answer will be from _delta_log."

Not exactly.

Here's what actually happens:

You run:
SELECT * FROM day3_employee;
Delta first reads _delta_log to determine which Parquet files belong to the latest version.
Then it reads those Parquet files.
Finally, it returns the data.

So:

✅ _delta_log = acts like an index or transaction history.
✅ Parquet files = contain the actual rows.
This is an interview-quality answer

If an interviewer asks:

Where is the data stored in a Delta table?

You can confidently say:

"The actual data is stored as Parquet files in the Data Lake. Delta Lake adds a _delta_log transaction log, also stored in the Data Lake, which tracks every change and enables ACID transactions, Time Travel, Schema Enforcement, and Schema Evolution."

That's an excellent answer and demonstrates you understand how Delta Lake works internally.